# 手撕 A2C (Advantage Actor-Critic)

## 背景
A2C = REINFORCE + critic baseline。Critic 估计 V(s)，优势 A(s,a) = r + γV(s') - V(s)。
用 critic 减方差，同步更新 actor 和 critic。

## 考察点
- advantage 的计算（TD error）
- actor loss = -log(π(a|s)) * advantage
- critic loss = MSE(V(s), r + γV(s'))
- 与 REINFORCE / PPO 的关系

In [ ]:
import torch
import torch.nn as nn

class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(),
                                   nn.Linear(hidden, act_dim))
        self.critic = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(),
                                    nn.Linear(hidden, 1))

    def get_action(self, obs):
        logits = self.actor(obs)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action)

    def get_value(self, obs):
        return self.critic(obs).squeeze(-1)

def a2c_update(model, optimizer, obs, action, reward, next_obs, done, gamma=0.99):
    value = model.get_value(obs)
    next_value = model.get_value(next_obs) * (1 - done)
    advantage = reward + gamma * next_value - value  # TD error as advantage
    # Actor loss
    logits = model.actor(obs)
    log_prob = torch.distributions.Categorical(logits=logits).log_prob(action)
    actor_loss = -(log_prob * advantage.detach()).mean()
    # Critic loss
    critic_loss = advantage.pow(2).mean()
    loss = actor_loss + 0.5 * critic_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
# 在简单 CartPole-like 环境验证
torch.manual_seed(42)
obs_dim, act_dim = 4, 2
model = ActorCritic(obs_dim, act_dim)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
losses = []
for _ in range(500):
    obs = torch.randn(obs_dim)
    action, _ = model.get_action(obs)
    reward = 1.0 if action == 0 else 0.5
    next_obs = torch.randn(obs_dim)
    done = torch.tensor(0.0)
    loss = a2c_update(model, opt, obs, action, reward, next_obs, done)
    losses.append(loss)
assert losses[-1] < losses[0], "loss 应下降"
print(f"初始 loss: {losses[0]:.4f} → 最终 loss: {losses[-1]:.4f}")
print("✅ A2C 训练 loss 下降")